# Day 29 — Building APIs

**How to use this notebook:**
- Read each exercise, fill in your solution (replace `pass`)
- Run the test cell right after to see **APPROVED** or **FAILED**
- Do NOT modify the test cells

**Key concepts today:** REST architecture, endpoint design, request/response formatting, CRUD over HTTP, status codes in context

> Exercises test the **business logic** that would sit inside Flask API endpoints — no server required.

---
## Exercise 1 — Rate Limiter

Every production API enforces **rate limiting** — restricting how many requests a user can make in a given time window. This protects the server from abuse and ensures fair usage.

Build a `RateLimiter` class that:

- `__init__(self, max_requests, window_seconds)` — stores the limit and time window
- `check(user_id)` — records the current timestamp for `user_id` and checks if they are within the limit:
  - First, remove any timestamps older than `window_seconds` from their history
  - If request count is **under** the limit: record the timestamp, return `{'allowed': True, 'remaining': n}`
  - If request count is **at or over** the limit: return `{'allowed': False, 'retry_after': seconds_until_oldest_expires}`
- `reset(user_id)` — clears all request history for that user

**Hint:** Use `datetime.now()` and `timedelta` to track and compare timestamps (Day 16). Store timestamps per user in a dict of lists.

In [ ]:
from datetime import datetime, timedelta

class RateLimiter:
    def __init__(self, max_requests, window_seconds):
        self.max_requests = max_requests
        self.window_seconds = window_seconds
        track_request_ts = []

    def check(self, user_id):
        current_time = datetime.now()

        if user_id not in self.track_request_ts:
            self.track_request_ts = []

        self.track_request_ts[user_id] = [ts for ts in self.track_request_ts[user_id] if (current_time - ts).total_seconds() < self.window_seconds ]

        if len(self.track_request_ts) >= self.max_requests:
            oldest_req = self.track_request_ts[0]
            retry_after = self.window_seconds - (current_time - oldest_req).total_seconds()
            return {'allowed': False, 'retry_after': retry_after}
        
        self.track_request_ts.append(self.max_requests)
        remaining = self.max_requests - len(self.track_request_ts)
        return {'allowed': True, 'remaining': remaining}

    def reset(self, user_id):
        self.track_request_ts[user_id] = []

In [ ]:
# TEST CELL — do not modify
import time

def _check(label, got, expected):
    if got == expected:
        print(f'  ✔ APPROVED — {label}')
    else:
        print(f'  ✘ FAILED   — {label}')
        print(f'             got:      {got!r}')
        print(f'             expected: {expected!r}')

print('\n── Exercise 1: RateLimiter ──')
limiter = RateLimiter(max_requests=3, window_seconds=10)

r1 = limiter.check('alice')
_check('1st request allowed',   r1.get('allowed'),    True)
_check('1st remaining',         r1.get('remaining'),  2)

r2 = limiter.check('alice')
_check('2nd request allowed',   r2.get('allowed'),    True)
_check('2nd remaining',         r2.get('remaining'),  1)

r3 = limiter.check('alice')
_check('3rd request allowed',   r3.get('allowed'),    True)
_check('3rd remaining',         r3.get('remaining'),  0)

r4 = limiter.check('alice')
_check('4th request blocked',   r4.get('allowed'),    False)
_check('retry_after present',   'retry_after' in r4,  True)

# Different user is independent
rb = limiter.check('bob')
_check('bob unaffected',        rb.get('allowed'),    True)

# Reset clears history
limiter.reset('alice')
r5 = limiter.check('alice')
_check('after reset allowed',   r5.get('allowed'),    True)
_check('after reset remaining', r5.get('remaining'),  2)

---
## Exercise 2 — Request Payload Validator

Before an API endpoint processes data it must validate the incoming payload.

Write `validate_payload(payload, schema)` where:
- `payload` is a `dict` of field names → values sent by the client.
- `schema` is a `dict` of field names → rule dicts. Each rule may contain:
  - `'required': True` — field must be present and not `None`
  - `'type': <type>` — value must be an instance of that type (e.g. `str`, `int`)
  - `'min': <number>` — for `int`/`float`: value must be `>= min`
  - `'max': <number>` — for `int`/`float`: value must be `<= max`

Return a dict:
```python
{'valid': bool, 'errors': list_of_error_strings}
```
- `valid` is `True` only when `errors` is empty.
- One error string per violated rule, e.g. `"'age' is required"`,
  `"'age' must be of type int"`, `"'age' must be >= 18"`.

In [ ]:
def validate_payload(payload, schema):
    pass

In [ ]:
# TEST CELL — do not modify
def _check(label, got, expected):
    if got == expected:
        print(f'  ✔ APPROVED — {label}')
    else:
        print(f'  ✘ FAILED   — {label}')
        print(f'             got:      {got!r}')
        print(f'             expected: {expected!r}')

print('\n── Exercise 2: validate_payload ──')
schema = {
    'name': {'required': True, 'type': str},
    'age':  {'required': True, 'type': int, 'min': 18, 'max': 120},
    'bio':  {'type': str},
}

ok = validate_payload({'name': 'Alice', 'age': 25}, schema)
_check('valid payload',   ok['valid'],  True)
_check('no errors',       ok['errors'], [])

missing = validate_payload({'age': 25}, schema)
_check('missing name valid',  missing['valid'],            False)
_check('missing name error',  "'name' is required" in missing['errors'], True)

wrong_type = validate_payload({'name': 'Bob', 'age': '30'}, schema)
_check('wrong type valid',    wrong_type['valid'],          False)
_check('wrong type error',    "'age' must be of type int" in wrong_type['errors'], True)

too_young = validate_payload({'name': 'Kid', 'age': 10}, schema)
_check('too young valid',     too_young['valid'],           False)
_check('too young error',     "'age' must be >= 18" in too_young['errors'], True)

too_old = validate_payload({'name': 'Elder', 'age': 200}, schema)
_check('too old valid',       too_old['valid'],             False)
_check('too old error',       "'age' must be <= 120" in too_old['errors'], True)

---
## Mini-Project — Full API Simulation

Combine everything: build a complete in-memory REST API for a **book library**.

Each book: `{'id': int, 'title': str, 'author': str, 'year': int, 'available': bool}`

```python
LibraryAPI class:
  books = []
  
  .add_book(title, author, year)    → 201 success response
  .get_books(author=None)           → 200 with filtered list (filter by author if provided)
  .borrow_book(book_id)             → 200 if available, 400 if already borrowed, 404 if not found
  .return_book(book_id)             → 200 if was borrowed, 400 if already available, 404 if not found
  .get_stats()                      → 200 with {'total': n, 'available': n, 'borrowed': n}
```

In [ ]:
class LibraryAPI:
    def __init__(self):
        self.books = []

    def add_book(self, title, author, year):
        pass

    def get_books(self, author=None):
        pass

    def borrow_book(self, book_id):
        pass

    def return_book(self, book_id):
        pass

    def get_stats(self):
        pass

In [ ]:
# TEST CELL — do not modify
def _check(label, got, expected):
    if got == expected:
        print(f'  ✔ APPROVED — {label}')
    else:
        print(f'  ✘ FAILED   — {label}')
        print(f'             got:      {got!r}')
        print(f'             expected: {expected!r}')

print('\n── Mini-Project: LibraryAPI ──')
lib = LibraryAPI()
lib.add_book('Python Crash Course','Eric Matthes',2019)
lib.add_book('Clean Code','Robert Martin',2008)
lib.add_book('Fluent Python','Luciano Ramalho',2015)

_check('total books', len(lib.books), 3)

all_books = lib.get_books()
_check('get_all status',  all_books.get('status'), 200)
_check('get_all count',   len(all_books.get('data',[])), 3)

filtered = lib.get_books(author='Eric Matthes')
_check('filtered count',  len(filtered.get('data',[])), 1)

borrow = lib.borrow_book(1)
_check('borrow status',   borrow.get('status'), 200)
_check('borrow again',    lib.borrow_book(1).get('status'), 400)
_check('borrow 404',      lib.borrow_book(99).get('status'), 404)

ret = lib.return_book(1)
_check('return status',   ret.get('status'), 200)
_check('return again',    lib.return_book(1).get('status'), 400)

stats = lib.get_stats().get('data',{})
_check('stats total',     stats.get('total'),     3)
_check('stats available', stats.get('available'), 3)
_check('stats borrowed',  stats.get('borrowed'),  0)